# Movie Recommendations with FlashInfer on AMD ROCm

End-to-end walkthrough of the **FlashInfer+ROCm** library for LLM-based recommendation:

| Section | What it shows |
|---|---|
| Part 1 | Tensor-level `single_prefill_with_kv_cache` and `single_decode_with_kv_cache` |
| Part 2 | Load MovieLens-100k, extract per-user viewing history |
| Part 3 | Load a small HuggingFace LLM (baseline) |
| Part 4 | Register FlashInfer as an HF attention backend |
| Part 5 | Correctness check: eager vs FlashInfer generation |
| Part 6 | Generate movie recommendations for real users |
| Part 7 | `BatchPrefillWithPagedKVCacheWrapper` tour |
| Part 8 | `BatchDecodeWithPagedKVCacheWrapper` tour |
| Part 9 | Benchmark FlashInfer batch APIs vs PyTorch SDPA |
| Part 10 | Integrated single-user latency vs SDPA (diagnostic) |

**Hardware:** AMD Instinct MI300X / MI325X / MI355X with ROCm 7.x. The easiest way to run the notebook is inside the official Docker image:

```bash
docker run -it --privileged --network=host --device=/dev/kfd --device=/dev/dri \
  --group-add video --cap-add=SYS_PTRACE --security-opt seccomp=unconfined \
  --ipc=host --shm-size 128G \
  rocm/flashinfer:flashinfer-0.5.3.amd1_rocm7.2_ubuntu24.04_py3.12_pytorch2.9.1
```

### FlashInfer ROCm constraints to remember

Before picking a model or shape, keep these limits in mind (they show up as runtime errors otherwise):

| Constraint | Values |
|---|---|
| KV layout | `NHD` (we stick to this throughout) |
| `head_dim` | 64, 128, 256, 512 |
| GQA `group_size` for `single_decode_with_kv_cache` | 1, 2, 3, 4, 8 (see `DISPATCH_GQA_GROUP_SIZE` in `include/flashinfer/attention/generic/dispatch.cuh`) |
| HF `attn_implementation` name | must **not** contain the substring `flash` — Transformers will try to load a Flash-Attention hub kernel otherwise |
| AITER backend | prefill only, `NHD` only, paged `page_size ∈ {1, 16, 1024}` |

We pick **SmolLM2-1.7B-Instruct** (MHA, `group_size=1`, `head_dim=64`) because it is non-gated and satisfies every constraint.

## 0. Install extra dependencies

In [1]:
!pip install -q "transformers>=4.45" accelerate pandas requests

### Environment check

In [2]:
import os

# Silence FlashInfer's per-call ROCm 'enable_pdl' / 'auto backend' warnings.
# They are informational and fire once per wrapper call, which can flood the
# benchmark cell below. Set before importing flashinfer.
os.environ["FLASHINFER_LOGGING_LEVEL"] = "error"

import torch
import flashinfer

print(f"PyTorch version  : {torch.__version__}")
print(f"HIP version      : {torch.version.hip}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"FlashInfer       : {flashinfer.__version__}")

[aiter] import [module_aiter_enum] under /opt/conda/envs/flashinfer-py3.12-torch2.9.1-rocm7.2/lib/python3.12/site-packages/aiter/jit/module_aiter_enum.so


PyTorch version  : 2.9.1+rocm7.2.0.git7e1940d4
HIP version      : 7.2.26015-fc0010cf6a
CUDA available   : True
GPU              : AMD Radeon Graphics
FlashInfer       : 0.5.3+amd.1.dev7


---
## Part 1 — Single-sequence FlashInfer APIs

Two primitives power everything a causal-LM does at inference time:

| API | When it runs | Input Q shape |
|---|---|---|
| `single_prefill_with_kv_cache` | Process the whole prompt at once | `[qo_len, num_qo_heads, head_dim]` |
| `single_decode_with_kv_cache`  | Generate one new token against the cached KV | `[num_qo_heads, head_dim]` |

We cross-check each one against a naive FP32 reference. We use **non-causal** attention here so the FP16 tolerances stay tight across backends; causal masking is exercised in Parts 4+.

In [3]:
import math


def naive_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = False) -> torch.Tensor:
    """Reference attention, NHD layout, GQA-aware."""
    qo_len, num_qo_heads, head_dim = q.shape
    kv_len, num_kv_heads, _ = k.shape
    sm_scale = 1.0 / math.sqrt(head_dim)

    group_size = num_qo_heads // num_kv_heads
    if group_size > 1:
        k = k.repeat_interleave(group_size, dim=1)
        v = v.repeat_interleave(group_size, dim=1)

    q_t, k_t, v_t = q.transpose(0, 1), k.transpose(0, 1), v.transpose(0, 1)
    scores = torch.matmul(q_t, k_t.transpose(1, 2)) * sm_scale
    if causal:
        mask = torch.tril(torch.ones(qo_len, kv_len, device=q.device, dtype=torch.bool), diagonal=(kv_len - qo_len))
        scores = scores.masked_fill(~mask.unsqueeze(0), float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    out = torch.matmul(attn, v_t)
    return out.transpose(0, 1)

### 1a. Single prefill

In [4]:
# Shapes chosen to match SmolLM2-1.7B-Instruct: MHA 32/32, head_dim 64.
qo_len, kv_len = 128, 128
num_qo_heads, num_kv_heads, head_dim = 32, 32, 64

q = torch.randn(qo_len, num_qo_heads, head_dim, device="cuda:0", dtype=torch.float16)
k = torch.randn(kv_len, num_kv_heads, head_dim, device="cuda:0", dtype=torch.float16)
v = torch.randn(kv_len, num_kv_heads, head_dim, device="cuda:0", dtype=torch.float16)

out_fi = flashinfer.single_prefill_with_kv_cache(q, k, v, causal=False, kv_layout="NHD")
out_ref = naive_attention(q.float(), k.float(), v.float(), causal=False).to(torch.float16)

torch.testing.assert_close(out_fi, out_ref, rtol=1e-2, atol=1e-2)
print(f"single_prefill_with_kv_cache output shape: {tuple(out_fi.shape)}")
print(f"Max abs diff vs reference: {(out_fi - out_ref).abs().max().item():.6f}")
print("PASS")

single_prefill_with_kv_cache output shape: (128, 32, 64)
Max abs diff vs reference: 0.000488
PASS


### 1b. Single decode

During autoregressive generation the query is a single vector (`qo_len=1`) that attends over the full cached KV.

In [5]:
kv_len_decode = 512
q_d = torch.randn(num_qo_heads, head_dim, device="cuda:0", dtype=torch.float16)
k_d = torch.randn(kv_len_decode, num_kv_heads, head_dim, device="cuda:0", dtype=torch.float16)
v_d = torch.randn(kv_len_decode, num_kv_heads, head_dim, device="cuda:0", dtype=torch.float16)

out_fi_d = flashinfer.single_decode_with_kv_cache(q_d, k_d, v_d, kv_layout="NHD")
out_ref_d = naive_attention(q_d.unsqueeze(0).float(), k_d.float(), v_d.float(), causal=False).squeeze(0).to(torch.float16)

torch.testing.assert_close(out_fi_d, out_ref_d, rtol=1e-2, atol=1e-2)
print(f"single_decode_with_kv_cache output shape: {tuple(out_fi_d.shape)}")
print(f"Max abs diff vs reference: {(out_fi_d - out_ref_d).abs().max().item():.6f}")
print("PASS")

single_decode_with_kv_cache output shape: (32, 64)
Max abs diff vs reference: 0.000008
PASS


---
## Part 2 — Load MovieLens-100k

100,000 ratings from 943 users on 1,682 movies (~5 MB). We parse the ratings (`u.data`) and movie titles (`u.item`) and expose a `top_rated_titles(user_id, k)` helper. A hard-coded fallback user history is used if the download fails.

In [6]:
import io
import os
import zipfile

import pandas as pd
import requests

ML_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = "/tmp/ml-100k"

FALLBACK_HISTORY = {
    1: [
        "Toy Story (1995)", "GoldenEye (1995)", "Twelve Monkeys (1995)",
        "Babe (1995)", "Dead Man Walking (1995)", "Seven (Se7en) (1995)",
        "Usual Suspects, The (1995)", "Pulp Fiction (1994)",
        "Shawshank Redemption, The (1994)", "Fargo (1996)",
    ],
}

try:
    if not os.path.isdir(DATA_DIR):
        print("Downloading MovieLens-100k ...")
        resp = requests.get(ML_URL, timeout=30)
        resp.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
            z.extractall("/tmp")
        print("Done.")
    else:
        print("MovieLens-100k already cached at", DATA_DIR)

    ratings = pd.read_csv(
        f"{DATA_DIR}/u.data", sep="\t",
        names=["user_id", "item_id", "rating", "timestamp"],
    )
    movies = pd.read_csv(
        f"{DATA_DIR}/u.item", sep="|", encoding="latin-1", header=None,
        usecols=[0, 1], names=["item_id", "title"],
    )
    data = ratings.merge(movies, on="item_id")
    _dataset_loaded = True
    print(f"Loaded {len(ratings):,} ratings across {ratings['user_id'].nunique()} users "
          f"and {ratings['item_id'].nunique()} movies.")
except Exception as exc:
    print(f"Could not download MovieLens ({exc}); using hard-coded fallback.")
    data = None
    _dataset_loaded = False


def top_rated_titles(user_id: int, k: int = 10) -> list[str]:
    """Return the *k* highest-rated movie titles for *user_id*."""
    if _dataset_loaded:
        user_data = data[data["user_id"] == user_id]
        top = user_data.sort_values(["rating", "timestamp"], ascending=[False, False])
        return top["title"].head(k).tolist()
    return FALLBACK_HISTORY.get(user_id, FALLBACK_HISTORY[1])[:k]


print("\nExample — User 1 top-rated movies:")
for i, t in enumerate(top_rated_titles(1), 1):
    print(f"  {i}. {t}")

MovieLens-100k already cached at /tmp/ml-100k
Loaded 100,000 ratings across 943 users and 1682 movies.

Example — User 1 top-rated movies:
  1. Delicatessen (1991)
  2. Truth About Cats & Dogs, The (1996)
  3. Kolya (1996)
  4. Crumb (1994)
  5. Gattaca (1997)
  6. Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
  7. Breaking the Waves (1996)
  8. Bound (1996)
  9. Contact (1997)
  10. Star Trek: The Wrath of Khan (1982)


---
## Part 3 — Load the LLM (eager baseline)

We use **`HuggingFaceTB/SmolLM2-1.7B-Instruct`** — a Llama-architecture model with `num_attention_heads = num_key_value_heads = 32` (group size 1) and `head_dim = 64`, which satisfies FlashInfer ROCm's GQA and head-dim constraints listed above. A smaller fallback is commented out for laptops with less VRAM.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
# Lighter fallback (GQA 15/5, group_size=3):
# MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="eager",
).to("cuda:0")
model_eager.eval()

cfg = model_eager.config
_head_dim = cfg.hidden_size // cfg.num_attention_heads
_group = cfg.num_attention_heads // cfg.num_key_value_heads
print(f"Model            : {MODEL_ID}")
print(f"Layers           : {cfg.num_hidden_layers}")
print(f"Query heads      : {cfg.num_attention_heads}")
print(f"KV heads         : {cfg.num_key_value_heads}")
print(f"Head dim         : {_head_dim}")
print(f"GQA group size   : {_group}")
print(f"Params           : {sum(p.numel() for p in model_eager.parameters()) / 1e6:.1f}M")

assert _head_dim in (64, 128, 256, 512), "FlashInfer ROCm head_dim must be 64/128/256/512"
assert _group in (1, 2, 3, 4, 8), "FlashInfer single_decode only supports group_size ∈ {1,2,3,4,8}"

/opt/conda/envs/flashinfer-py3.12-torch2.9.1-rocm7.2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`torch_dtype` is deprecated! Use `dtype` instead!



Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]


Loading weights:  76%|███████▌  | 165/218 [00:00<00:00, 1606.43it/s]


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 1605.11it/s]

Model            : HuggingFaceTB/SmolLM2-1.7B-Instruct
Layers           : 24
Query heads      : 32
KV heads         : 32
Head dim         : 64
GQA group size   : 1
Params           : 1711.4M


---
## Part 4 — Register FlashInfer as an HF attention backend

Transformers `>=4.45` looks up attention implementations in `ALL_ATTENTION_FUNCTIONS`. We register a callback that dispatches to `single_prefill_with_kv_cache` for the prompt phase and to `single_decode_with_kv_cache` for each generated token.

**Shape contract** (matches `transformers.models.llama.modeling_llama.eager_attention_forward`):

* Inputs:  `query (B, H_q, L, D)`, `key (B, H_kv, L_kv, D)`, `value (B, H_kv, L_kv, D)`
* Output: `attn_output (B, L, H_q, D)`  (note — heads axis is **2**, not 1; returning `(B, H_q, L, D)` silently breaks `o_proj`)

**Constraints of this backend:** batch size 1, `NHD` layout, causal prefill, no sliding window, fp16/bf16. The name is `fi_rocm` because Transformers treats any string containing `flash` as Flash-Attention and tries to load a hub kernel.

In [8]:
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS


def flashinfer_attention(module, query, key, value, attention_mask,
                         scaling=None, dropout=0.0, **kwargs):
    bsz = query.shape[0]
    assert bsz == 1, "fi_rocm backend only supports batch_size=1"

    # (1, H, L, D) → (L, H, D) for NHD layout.
    q = query[0].transpose(0, 1).contiguous()
    k = key[0].transpose(0, 1).contiguous()
    v = value[0].transpose(0, 1).contiguous()

    sm_scale = scaling if scaling is not None else q.shape[-1] ** -0.5

    if q.shape[0] > 1:
        # Prompt phase: causal prefill over the whole prompt.
        out = flashinfer.single_prefill_with_kv_cache(
            q, k, v, causal=True, kv_layout="NHD", sm_scale=sm_scale,
        )  # (L, H, D)
        attn_output = out.unsqueeze(0).contiguous()          # (1, L, H, D)
    else:
        # Decode phase: one query token against the full cached KV.
        dec = flashinfer.single_decode_with_kv_cache(
            q.squeeze(0), k, v, kv_layout="NHD", sm_scale=sm_scale,
        )  # (H, D)
        attn_output = dec.unsqueeze(0).unsqueeze(0).contiguous()  # (1, 1, H, D)

    return attn_output, None


ALL_ATTENTION_FUNCTIONS["fi_rocm"] = flashinfer_attention
print("Registered 'fi_rocm' in ALL_ATTENTION_FUNCTIONS")

Registered 'fi_rocm' in ALL_ATTENTION_FUNCTIONS


In [9]:
del model_eager
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="fi_rocm",
).to("cuda:0")
model.eval()
print("Model loaded with attn_implementation='fi_rocm' (FlashInfer-backed).")


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]


Loading weights:  71%|███████   | 154/218 [00:00<00:00, 1539.63it/s]


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 1531.04it/s]

Model loaded with attn_implementation='fi_rocm' (FlashInfer-backed).


---
## Part 5 — Correctness check: eager vs FlashInfer

We generate the same chat prompt with `attn_implementation="eager"` and with `fi_rocm` and verify the first N tokens agree (FP16 rounding can occasionally flip one late token, so we require ≥ 90% overlap rather than exact equality).

In [10]:
def _chat_ids(prompt_text: str):
    msgs = [{"role": "user", "content": prompt_text}]
    s = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(s, return_tensors="pt")
    return enc["input_ids"].to("cuda:0"), enc["attention_mask"].to("cuda:0")


def _greedy_tokens(m, ids, mask, n):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    with torch.no_grad():
        out = m.generate(
            ids,
            attention_mask=mask,
            max_new_tokens=n,
            do_sample=False,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return out[0, ids.shape[1] :]


N = 32
sanity_prompt = "What is the capital of France? Answer in one short English sentence."
ids, mask = _chat_ids(sanity_prompt)

# Eager baseline
m_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="eager",
).to("cuda:0")
m_eager.eval()
ids_eager = _greedy_tokens(m_eager, ids, mask, N)
del m_eager
torch.cuda.empty_cache()

ids_fi = _greedy_tokens(model, ids, mask, N)

overlap = (ids_eager[: min(len(ids_eager), len(ids_fi))] == ids_fi[: min(len(ids_eager), len(ids_fi))]).float().mean().item()
print(f"Eager  : {tokenizer.decode(ids_eager, skip_special_tokens=True).strip()!r}")
print(f"fi_rocm: {tokenizer.decode(ids_fi, skip_special_tokens=True).strip()!r}")
print(f"Token-level agreement over first {N} tokens: {overlap*100:.1f}%")
assert overlap >= 0.9, "FlashInfer output disagrees too much with eager; check the attention shape contract."
print("PASS")


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]


Loading weights:  67%|██████▋   | 147/218 [00:00<00:00, 1436.47it/s]


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 1505.00it/s]

Eager  : 'The capital of France is Paris.'
fi_rocm: 'The capital of France is Paris.'
Token-level agreement over first 32 tokens: 100.0%
PASS


---
## Part 6 — Movie recommendations

We build a chat prompt from the user's top-rated movies and ask the model for 5 new suggestions. Every attention call — prompt prefill plus each decode step — goes through FlashInfer.

In [11]:
SYSTEM_PROMPT = (
    "You are a movie recommendation assistant. Given a list of movies a user enjoyed, "
    "suggest 5 different movies they would like. For each suggestion give the title "
    "followed by a one-sentence reason. Reply in English only."
)


def build_prompt(liked_titles: list[str]) -> str:
    titles = ", ".join(liked_titles)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"I really enjoyed these movies: {titles}\n\nWhat should I watch next?"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_recommendations(liked_titles: list[str], max_new_tokens: int = 192) -> str:
    prompt = build_prompt(liked_titles)
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to("cuda:0")
    attention_mask = enc["attention_mask"].to("cuda:0")
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    with torch.no_grad():
        out = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, input_ids.shape[1] :], skip_special_tokens=True).strip()

In [12]:
user_id = 1
liked = top_rated_titles(user_id, k=10)

print(f"=== User {user_id} — liked movies ===")
for i, t in enumerate(liked, 1):
    print(f"  {i}. {t}")

recs = generate_recommendations(liked)
print("\n=== Recommendations (FlashInfer ROCm) ===")
print(recs)
assert len(recs) > 50, "Recommendation text looks empty/truncated."

=== User 1 — liked movies ===
  1. Delicatessen (1991)
  2. Truth About Cats & Dogs, The (1996)
  3. Kolya (1996)
  4. Crumb (1994)
  5. Gattaca (1997)
  6. Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
  7. Breaking the Waves (1996)
  8. Bound (1996)
  9. Contact (1997)
  10. Star Trek: The Wrath of Khan (1982)



=== Recommendations (FlashInfer ROCm) ===
1. The Shawshank Redemption (1994)
Reason: The film's uplifting and inspiring storyline will resonate with you.

2. Pulp Fiction (1994)
Reason: The film's unique storytelling and memorable characters will captivate you.

3. The Godfather (1972)
Reason: The film's timeless themes and iconic characters will leave a lasting impression.

4. The Dark Knight (2008)
Reason: The film's dark and thrilling storyline will keep you on the edge of your seat.

5. The Lord of the Rings: The Fellowship of the Ring (2001)
Reason: The film's epic storyline and stunning visuals will transport you to a new world.


### Recommendations for a handful of users

In [13]:
import random

random.seed(42)
if _dataset_loaded:
    all_users = sorted(data["user_id"].unique())
else:
    all_users = list(FALLBACK_HISTORY.keys())

sample_users = random.sample(all_users, min(3, len(all_users)))

rows = []
for uid in sample_users:
    liked = top_rated_titles(uid, k=8)
    recs = generate_recommendations(liked, max_new_tokens=160)
    rows.append({
        "user_id": uid,
        "liked_titles": ", ".join(liked[:4]) + (" ..." if len(liked) > 4 else ""),
        "recommendations": recs[:400],
    })
    print(f"User {uid} done.")

pd.DataFrame(rows)

User 655 done.


User 115 done.


User 26 done.


,user_id,liked_titles,recommendations
0,655,"In the Company of Men (1997), Belle de jour (1...",I would recommend the following movies:\n1. Th...
1,115,"12 Angry Men (1957), Man Who Would Be King, Th...","I would recommend watching ""The Godfather"" (19..."
2,26,"Fargo (1996), Godfather, The (1972), L.A. Conf...",Argo (1997)\n\nReason: Argo is a thrilling pol...


---
## Part 7 — Batch prefill with paged KV

`BatchPrefillWithPagedKVCacheWrapper` fuses many sequences into one kernel launch and reads the KV cache through a page table. This is the core primitive used by continuous-batching serving engines such as vLLM/SGLang.

The inputs are:

* **`q`** — flat `[sum(qo_len_i), H_q, D]` with query indptr.
* **Paged KV** — `[num_pages, 2, page_size, H_kv, D]` (NHD) + `kv_indptr`, `kv_indices`, `kv_last_page_len`.

We validate every row of the batched output against `single_prefill_with_kv_cache` on the un-paged slice.

In [14]:
def run_batch_prefill(batch_size: int, qo_len: int, kv_len: int, page_size: int,
                     num_qo_heads: int, num_kv_heads: int, head_dim: int,
                     causal: bool = True):
    device = "cuda:0"
    dtype = torch.float16

    q = torch.randn(batch_size * qo_len, num_qo_heads, head_dim, device=device, dtype=dtype)
    q_indptr = torch.arange(0, batch_size + 1, device=device, dtype=torch.int32) * qo_len

    num_pages_per_seq = (kv_len + page_size - 1) // page_size
    total_num_pages = num_pages_per_seq * batch_size
    kv_data = torch.randn(
        total_num_pages, 2, page_size, num_kv_heads, head_dim, device=device, dtype=dtype,
    )
    kv_indptr = torch.arange(0, batch_size + 1, device=device, dtype=torch.int32) * num_pages_per_seq
    kv_indices = torch.arange(0, total_num_pages, device=device, dtype=torch.int32)
    kv_last_page_len = torch.full(
        (batch_size,), (kv_len - 1) % page_size + 1, dtype=torch.int32, device=device,
    )

    workspace = torch.empty(256 * 1024 * 1024, dtype=torch.int8, device=device)
    wrapper = flashinfer.prefill.BatchPrefillWithPagedKVCacheWrapper(workspace, "NHD")
    wrapper.plan(
        q_indptr, kv_indptr, kv_indices, kv_last_page_len,
        num_qo_heads, num_kv_heads, head_dim, page_size, causal=causal,
    )
    out = wrapper.run(q, kv_data)
    return q, kv_data, q_indptr, kv_indptr, kv_indices, kv_last_page_len, out


BATCH_SIZE = 4
QO_LEN = 256
KV_LEN = 256
PAGE_SIZE = 16
H_Q = cfg.num_attention_heads
H_KV = cfg.num_key_value_heads
D = cfg.hidden_size // cfg.num_attention_heads

q, kv_data, q_indptr, kv_indptr, kv_indices, kv_last_page_len, out_batch = run_batch_prefill(
    BATCH_SIZE, QO_LEN, KV_LEN, PAGE_SIZE, H_Q, H_KV, D, causal=True,
)
print(f"Batched output shape: {tuple(out_batch.shape)} (sum(qo_len), H, D)")

# Row-by-row verification against single_prefill_with_kv_cache on the un-paged KV.
q_cpu_indptr = q_indptr.cpu()
kv_cpu_indptr = kv_indptr.cpu()
kv_cpu_last = kv_last_page_len.cpu()
max_err = 0.0
for i in range(BATCH_SIZE):
    qi = q[q_cpu_indptr[i] : q_cpu_indptr[i + 1]]
    pages = kv_data[kv_cpu_indptr[i] : kv_cpu_indptr[i + 1]]
    # Stitch the paged KV for this sequence back into [kv_len, H_kv, D].
    full_k = pages[:-1, 0].reshape(-1, H_KV, D)
    full_v = pages[:-1, 1].reshape(-1, H_KV, D)
    last_k = pages[-1, 0, : int(kv_cpu_last[i])].reshape(-1, H_KV, D)
    last_v = pages[-1, 1, : int(kv_cpu_last[i])].reshape(-1, H_KV, D)
    ki = torch.cat([full_k, last_k], dim=0)
    vi = torch.cat([full_v, last_v], dim=0)
    ref = flashinfer.single_prefill_with_kv_cache(qi, ki, vi, causal=True, kv_layout="NHD")
    got = out_batch[q_cpu_indptr[i] : q_cpu_indptr[i + 1]]
    err = (ref - got).abs().max().item()
    max_err = max(max_err, err)
print(f"Batch-vs-single max abs diff across {BATCH_SIZE} rows: {max_err:.4e}")
assert max_err < 5e-2, "Batch prefill deviates too much from single prefill."
print("PASS")

Batched output shape: (1024, 32, 64) (sum(qo_len), H, D)
Batch-vs-single max abs diff across 4 rows: 0.0000e+00
PASS


---
## Part 8 — Batch decode with paged KV

`BatchDecodeWithPagedKVCacheWrapper` is the decode-phase counterpart — one query token per sequence, reading through the same paged KV layout.

In [15]:
def run_batch_decode(batch_size: int, kv_len: int, page_size: int,
                    num_qo_heads: int, num_kv_heads: int, head_dim: int):
    device = "cuda:0"
    dtype = torch.float16

    q = torch.randn(batch_size, num_qo_heads, head_dim, device=device, dtype=dtype)

    num_pages_per_seq = (kv_len + page_size - 1) // page_size
    total_num_pages = num_pages_per_seq * batch_size
    kv_data = torch.randn(
        total_num_pages, 2, page_size, num_kv_heads, head_dim, device=device, dtype=dtype,
    )
    kv_indptr = torch.arange(0, batch_size + 1, device=device, dtype=torch.int32) * num_pages_per_seq
    kv_indices = torch.arange(0, total_num_pages, device=device, dtype=torch.int32)
    kv_last_page_len = torch.full(
        (batch_size,), (kv_len - 1) % page_size + 1, dtype=torch.int32, device=device,
    )

    workspace = torch.empty(128 * 1024 * 1024, dtype=torch.int8, device=device)
    wrapper = flashinfer.decode.BatchDecodeWithPagedKVCacheWrapper(workspace, "NHD")
    wrapper.plan(
        kv_indptr, kv_indices, kv_last_page_len,
        num_qo_heads, num_kv_heads, head_dim, page_size,
        data_type=dtype, q_data_type=dtype,
    )
    out = wrapper.run(q, kv_data)
    return q, kv_data, kv_indptr, kv_indices, kv_last_page_len, out


DEC_BATCH = 4
DEC_KV = 512
DEC_PAGE = 16

q_d, kv_data_d, kv_indptr_d, kv_indices_d, kv_last_d, out_dec = run_batch_decode(
    DEC_BATCH, DEC_KV, DEC_PAGE, H_Q, H_KV, D,
)
print(f"Batched decode output shape: {tuple(out_dec.shape)}  (batch, H_q, D)")

# Reference: single_decode_with_kv_cache row-by-row on the un-paged KV.
kv_indptr_cpu = kv_indptr_d.cpu()
kv_last_cpu = kv_last_d.cpu()
max_err = 0.0
for i in range(DEC_BATCH):
    pages = kv_data_d[kv_indptr_cpu[i] : kv_indptr_cpu[i + 1]]
    full_k = pages[:-1, 0].reshape(-1, H_KV, D)
    full_v = pages[:-1, 1].reshape(-1, H_KV, D)
    last_k = pages[-1, 0, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
    last_v = pages[-1, 1, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
    ki = torch.cat([full_k, last_k], dim=0)
    vi = torch.cat([full_v, last_v], dim=0)
    ref = flashinfer.single_decode_with_kv_cache(q_d[i], ki, vi, kv_layout="NHD")
    err = (ref - out_dec[i]).abs().max().item()
    max_err = max(max_err, err)
print(f"Batch-vs-single max abs diff across {DEC_BATCH} rows: {max_err:.4e}")
assert max_err < 5e-2, "Batch decode deviates too much from single decode."
print("PASS")

Batched decode output shape: (4, 32, 64)  (batch, H_q, D)
Batch-vs-single max abs diff across 4 rows: 2.4414e-04
PASS


---
## Part 9 — Benchmark: FlashInfer batch vs naive PyTorch SDPA

In continuous-batching serving the per-step cost is dominated by (a) the fused prefill kernel and (b) the decode kernel that reads the large cached KV. Here we compare FlashInfer's batch wrappers against a simple PyTorch baseline that calls `torch.nn.functional.scaled_dot_product_attention` once per sequence in a Python loop — the same shape of workload a handwritten PyTorch server would issue.

Shapes follow the model above (`H_q = H_kv = 32`, `D = 64`) with batch 32 — a realistic continuous-batching size where FlashInfer's fused kernel amortises per-launch overhead that the SDPA-per-row loop pays on every sequence.

In [16]:
import time


def bench(fn, iters=50, warmup=10):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters  # ms/iter


def sdpa_baseline_prefill(q_flat, kv_data_p, q_indptr_cpu, kv_indptr_cpu, kv_last_cpu, B, kv_total_len):
    outs = []
    for i in range(B):
        qi = q_flat[q_indptr_cpu[i] : q_indptr_cpu[i + 1]]               # (L_q, H, D)
        pages = kv_data_p[kv_indptr_cpu[i] : kv_indptr_cpu[i + 1]]
        full_k = pages[:-1, 0].reshape(-1, H_KV, D)
        full_v = pages[:-1, 1].reshape(-1, H_KV, D)
        last_k = pages[-1, 0, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
        last_v = pages[-1, 1, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
        ki = torch.cat([full_k, last_k], dim=0)
        vi = torch.cat([full_v, last_v], dim=0)
        qi_sdpa = qi.transpose(0, 1).unsqueeze(0)   # (1, H, L_q, D)
        ki_sdpa = ki.transpose(0, 1).unsqueeze(0)
        vi_sdpa = vi.transpose(0, 1).unsqueeze(0)
        o = torch.nn.functional.scaled_dot_product_attention(qi_sdpa, ki_sdpa, vi_sdpa, is_causal=True)
        outs.append(o.squeeze(0).transpose(0, 1))
    return torch.cat(outs, dim=0)


def sdpa_baseline_decode(q_bhd, kv_data_p, kv_indptr_cpu, kv_last_cpu, B):
    outs = []
    for i in range(B):
        pages = kv_data_p[kv_indptr_cpu[i] : kv_indptr_cpu[i + 1]]
        full_k = pages[:-1, 0].reshape(-1, H_KV, D)
        full_v = pages[:-1, 1].reshape(-1, H_KV, D)
        last_k = pages[-1, 0, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
        last_v = pages[-1, 1, : int(kv_last_cpu[i])].reshape(-1, H_KV, D)
        ki = torch.cat([full_k, last_k], dim=0)
        vi = torch.cat([full_v, last_v], dim=0)
        qi_sdpa = q_bhd[i].unsqueeze(0).unsqueeze(2)       # (1, H, 1, D)
        ki_sdpa = ki.transpose(0, 1).unsqueeze(0)
        vi_sdpa = vi.transpose(0, 1).unsqueeze(0)
        o = torch.nn.functional.scaled_dot_product_attention(qi_sdpa, ki_sdpa, vi_sdpa, is_causal=False)
        outs.append(o.squeeze(0).squeeze(1))
    return torch.stack(outs, dim=0)

In [17]:
# --- Batch prefill: batch=32, qo_len=kv_len=256, page=16 ---
# Larger batch with shorter per-row work emphasises FlashInfer's batched
# kernel scheduling vs the Python-level SDPA-per-row launch overhead.
BP_B, BP_L, BP_PAGE = 32, 256, 16
q_bp, kv_bp, qip_bp, kvip_bp, kvidx_bp, kvlp_bp, _ = run_batch_prefill(
    BP_B, BP_L, BP_L, BP_PAGE, H_Q, H_KV, D, causal=True,
)
workspace = torch.empty(256 * 1024 * 1024, dtype=torch.int8, device="cuda:0")
wrap_bp = flashinfer.prefill.BatchPrefillWithPagedKVCacheWrapper(workspace, "NHD")
wrap_bp.plan(qip_bp, kvip_bp, kvidx_bp, kvlp_bp, H_Q, H_KV, D, BP_PAGE, causal=True)

qip_cpu = qip_bp.cpu()
kvip_cpu = kvip_bp.cpu()
kvlp_cpu = kvlp_bp.cpu()

ms_fi_prefill = bench(lambda: wrap_bp.run(q_bp, kv_bp))
ms_sdpa_prefill = bench(lambda: sdpa_baseline_prefill(q_bp, kv_bp, qip_cpu, kvip_cpu, kvlp_cpu, BP_B, BP_L))

# --- Batch decode: batch=32, kv_len=2048, page=16 ---
BD_B, BD_KV, BD_PAGE = 32, 2048, 16
q_bd, kv_bd, kvip_bd, kvidx_bd, kvlp_bd, _ = run_batch_decode(
    BD_B, BD_KV, BD_PAGE, H_Q, H_KV, D,
)
workspace_d = torch.empty(128 * 1024 * 1024, dtype=torch.int8, device="cuda:0")
wrap_bd = flashinfer.decode.BatchDecodeWithPagedKVCacheWrapper(workspace_d, "NHD")
wrap_bd.plan(
    kvip_bd, kvidx_bd, kvlp_bd, H_Q, H_KV, D, BD_PAGE,
    data_type=torch.float16, q_data_type=torch.float16,
)
kvip_cpu_d = kvip_bd.cpu()
kvlp_cpu_d = kvlp_bd.cpu()

ms_fi_decode = bench(lambda: wrap_bd.run(q_bd, kv_bd))
ms_sdpa_decode = bench(lambda: sdpa_baseline_decode(q_bd, kv_bd, kvip_cpu_d, kvlp_cpu_d, BD_B))

bench_df = pd.DataFrame(
    {
        "workload": [
            f"batch prefill (B={BP_B}, L={BP_L})",
            f"batch decode  (B={BD_B}, KV={BD_KV})",
        ],
        "FlashInfer (ms)": [ms_fi_prefill, ms_fi_decode],
        "PyTorch SDPA loop (ms)": [ms_sdpa_prefill, ms_sdpa_decode],
        "Speedup": [
            ms_sdpa_prefill / ms_fi_prefill,
            ms_sdpa_decode / ms_fi_decode,
        ],
    }
)
print("Batch-API benchmark")
print(bench_df.to_string(index=False, formatters={
    "FlashInfer (ms)": "{:.3f}".format,
    "PyTorch SDPA loop (ms)": "{:.3f}".format,
    "Speedup": "{:.2f}x".format,
}))

# Clean up workspace buffers to free VRAM for Part 10.
del q_bp, kv_bp, q_bd, kv_bd, wrap_bp, wrap_bd, workspace, workspace_d
torch.cuda.empty_cache()

Batch-API benchmark
                     workload FlashInfer (ms) PyTorch SDPA loop (ms) Speedup
  batch prefill (B=32, L=256)           0.376                  1.685   4.49x
batch decode  (B=32, KV=2048)           0.592                  3.118   5.27x


---
## Part 10 — Integrated single-user latency: FlashInfer vs SDPA

This is a **diagnostic** cell, not a fair fight. At batch size 1 the HuggingFace `generate()` loop dispatches one attention call per layer per decoded token through Python — the attention kernel itself is a tiny fraction of total time, so backend-to-backend speedups at this scale are dominated by Python/PyTorch dispatch cost. The place where FlashInfer actually shines is the batched/paged-KV path in Part 9.

In [18]:
bench_liked = top_rated_titles(1, k=10)
bench_prompt = build_prompt(bench_liked)
bench_enc = tokenizer(bench_prompt, return_tensors="pt")
bench_ids = bench_enc["input_ids"].to("cuda:0")
bench_mask = bench_enc["attention_mask"].to("cuda:0")
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
max_new = 128
gen_kw = dict(
    attention_mask=bench_mask, max_new_tokens=max_new, do_sample=False,
    pad_token_id=pad_id, eos_token_id=tokenizer.eos_token_id,
)


def _time_generate(m, ids):
    with torch.no_grad():
        _ = m.generate(ids, **gen_kw)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = m.generate(ids, **gen_kw)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return elapsed, out.shape[1] - ids.shape[1]


fi_time, fi_tokens = _time_generate(model, bench_ids)
print(f"FlashInfer (fi_rocm) : {fi_time*1000:.1f} ms total, "
      f"{fi_time/fi_tokens*1000:.2f} ms/token ({fi_tokens} tokens)")

del model
torch.cuda.empty_cache()

model_sdpa = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa",
).to("cuda:0")
model_sdpa.eval()
sdpa_time, sdpa_tokens = _time_generate(model_sdpa, bench_ids)
print(f"PyTorch SDPA         : {sdpa_time*1000:.1f} ms total, "
      f"{sdpa_time/sdpa_tokens*1000:.2f} ms/token ({sdpa_tokens} tokens)")
print(f"Per-token ratio (SDPA / FlashInfer): "
      f"{(sdpa_time/sdpa_tokens) / (fi_time/fi_tokens):.2f}x")

del model_sdpa
torch.cuda.empty_cache()

FlashInfer (fi_rocm) : 1102.7 ms total, 8.61 ms/token (128 tokens)



Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]


Loading weights:  61%|██████▏   | 134/218 [00:00<00:00, 1329.23it/s]


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 1423.67it/s]

PyTorch SDPA         : 1011.5 ms total, 7.90 ms/token (128 tokens)
Per-token ratio (SDPA / FlashInfer): 0.92x


---
## Summary

1. **Tensor-level APIs** — `single_prefill_with_kv_cache` and `single_decode_with_kv_cache` match a FP32 reference within FP16 tolerances.
2. **HF integration** — registering `fi_rocm` in `ALL_ATTENTION_FUNCTIONS` is enough to route every Llama-family attention call through FlashInfer; generated output matches the `eager` backend token-for-token.
3. **Recommendations** — end-to-end, plain `model.generate()` on real MovieLens users.
4. **Batched primitives** — `BatchPrefillWithPagedKVCacheWrapper` and `BatchDecodeWithPagedKVCacheWrapper` are validated row-by-row against the single-sequence APIs, and in Part 9 they beat a PyTorch SDPA loop on realistic serving shapes.

### Next steps

* Try `flashinfer.prefill.BatchPrefillWithRaggedKVCacheWrapper` for the case where the full KV lives in a single packed tensor (no pages).
* Swap `backend="aiter"` in `single_prefill_with_kv_cache` / `BatchPrefillWithPagedKVCacheWrapper` for an additional ROCm backend (see README "AITER Support").
* Move to a larger model whose GQA group size is still in `{1,2,3,4,8}` (for example, Llama-3 8B has group size 4 and `head_dim=128`, both supported).